## Import libraries for visualization

In [1]:
import pandas as pd
import folium
import matplotlib.pyplot as plt

In [2]:
# Import folium MarkerCluster plugin
from folium.plugins import MarkerCluster
# Import folium MousePosition plugin
from folium.plugins import MousePosition
# Import folium DivIcon plugin
from folium.features import DivIcon

## Load the dataset collected from earlier

In [3]:
filepath = '/Users/frederickhuang/Documents/VSCode/PY/SpaceX Rocket Launches/api_data_with_outcome_labels.csv'
df = pd.read_csv(filepath)
df.head()


,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude,Class
0,1,2010-06-04,Falcon 9,6123.547647,LEO,CCSFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857,0
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCSFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857,0
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCSFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857,0
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093,0
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCSFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857,0


## Create map object and dataframe that we'll need to plot markers and locations with.

In [36]:
# Start location is NASA Johnson Space Center
nasa_coordinate = [29.559684888503615, -95.0830971930759]
# Create map object starting at NASA JSC
m = folium.Map(location = nasa_coordinate, zoom_start=  5)

spacex_df = df[['LaunchSite', 'Latitude', 'Longitude', 'Class']]
launch_sites_df = spacex_df.groupby(['LaunchSite'], as_index=False).first()
launch_sites_df = launch_sites_df[['LaunchSite', 'Latitude', 'Longitude']]
launch_sites_df

,LaunchSite,Latitude,Longitude
0,CCSFS SLC 40,28.561857,-80.577366
1,KSC LC 39A,28.608058,-80.603956
2,VAFB SLC 4E,34.632093,-120.610829


## Create map labels

In [37]:
marker_cluster = MarkerCluster().add_to(m)
for lat, long, name in zip(launch_sites_df['Latitude'], launch_sites_df['Longitude'], launch_sites_df['LaunchSite']):
    circle = folium.Circle(location = [lat, long], radius=1000, color='#d35400', fill=True).add_child(folium.Popup(name))

    marker = folium.map.Marker(
    [lat, long],
    # Create an icon as a text label
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % name,
        )
    )
    m.add_child(marker)
    m.add_child(circle)
m

## Assign color labels corresponding to successful or failed rocket landing

In [38]:
colors = []
for i in range(0, spacex_df.shape[0]):
    if spacex_df['Class'][i] == 1:
        colors.append('green')
    else:
        colors.append('red')
spacex_df['marker_color'] = colors
spacex_df.head()

/var/folders/nv/brc1sfvj70n58vqs4_3_5m8h0000gn/T/ipykernel_40087/1377530405.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  spacex_df['marker_color'] = colors


,LaunchSite,Latitude,Longitude,Class,marker_color
0,CCSFS SLC 40,28.561857,-80.577366,0,red
1,CCSFS SLC 40,28.561857,-80.577366,0,red
2,CCSFS SLC 40,28.561857,-80.577366,0,red
3,VAFB SLC 4E,34.632093,-120.610829,0,red
4,CCSFS SLC 40,28.561857,-80.577366,0,red


## Plot colored label on map at each of the Falcon 9 launch sites based on launch outcome

In [39]:
for lat, long, number, color in zip(spacex_df['Latitude'], spacex_df['Longitude'], spacex_df['Class'], spacex_df['marker_color']):
    marker = folium.Marker(location = [lat, long], 
    
        icon = folium.Icon(color = color)
                          )
    marker_cluster.add_child(marker)
    
    # print(record)

m

## Add mouse position

In [41]:
''' Add Mouse Position to get the coordinate (Lat, Long) for a mouse over on the map
 this is mostly for if you want to calculate distances on this map to key geographic features later on
 to understand underlying characteristics of our launch sites '''
 
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(
    position='topright',
    separator=' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=20,
    prefix='Lat:',
    lat_formatter=formatter,
    lng_formatter=formatter,
)

m.add_child(mouse_position)
m